script/notebook block που διαβάζει το clean_dataset.csv, αφαιρεί τα mid, κάνει journal-year-label stratified cumulative sampling και παράγει filelists ανά στάδιο.

In [ ]:
""" Προσάρτηση του Google Drive """
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np

# =========================
# PATHS
# =========================

BASE_DIR = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER"
CLEAN_DATASET_PATH = os.path.join(BASE_DIR, "clean_dataset.csv")

OUT_DIR = os.path.join(BASE_DIR, "coverage_sampling_plan")
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# SETTINGS
# =========================

MILESTONES = {
    "v1_coverage_30pct": 0.30,
    "v2_coverage_50pct": 0.50,
    "v3_coverage_70pct": 0.70,
    "v4_coverage_100pct": 1.00
}

RANDOM_STATE = 42

# =========================
# LOAD DATA
# =========================

df = pd.read_csv(CLEAN_DATASET_PATH)

# κρατάμε μόνο low/high
df = df[df["citation_bin"].isin(["low", "high"])].copy()

# ασφάλεια τύπων
df["filename"] = df["filename"].astype(str)
df["journal"] = df["journal"].astype(str)
df["year"] = df["year"].astype(int)
df["citation_bin"] = df["citation_bin"].astype(str)

if "journal_short" not in df.columns:
    df["journal_short"] = df["journal"]

# stable random order μέσα σε κάθε stratum
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Total low/high papers:", len(df))
print(df.groupby("journal").size())

# =========================
# CREATE CUMULATIVE SAMPLING PLAN
# =========================

selected_rows = []

for journal, journal_df in df.groupby("journal", sort=True):

    journal_df = journal_df.copy()
    journal_total = len(journal_df)

    previous_selected = set()

    for milestone_name, pct in MILESTONES.items():

        target_total = int(round(journal_total * pct))

        if pct == 1.00:
            target_total = journal_total

        already_n = len(previous_selected)
        need_n = target_total - already_n

        if need_n <= 0:
            continue

        remaining_df = journal_df[~journal_df["filename"].isin(previous_selected)].copy()

        # stratified by year + citation_bin
        parts = []

        for (year, label), group in remaining_df.groupby(["year", "citation_bin"], sort=True):

            original_group_size = len(journal_df[
                (journal_df["year"] == year) &
                (journal_df["citation_bin"] == label)
            ])

            target_group_total = int(round(original_group_size * pct))

            already_group_n = sum(
                1 for f in previous_selected
                if f in set(journal_df[
                    (journal_df["year"] == year) &
                    (journal_df["citation_bin"] == label)
                ]["filename"])
            )

            need_group_n = target_group_total - already_group_n
            need_group_n = max(0, min(need_group_n, len(group)))

            if need_group_n > 0:
                sampled = group.sample(n=need_group_n, random_state=RANDOM_STATE)
                parts.append(sampled)

        if parts:
            new_selected_df = pd.concat(parts, ignore_index=True)
        else:
            new_selected_df = pd.DataFrame(columns=journal_df.columns)

        # Αν λόγω rounding δεν φτάσαμε το target_total, συμπληρώνουμε από τα υπόλοιπα
        current_add_n = len(new_selected_df)
        remaining_needed = need_n - current_add_n

        if remaining_needed > 0:
            already_new = set(new_selected_df["filename"]) if not new_selected_df.empty else set()

            filler_pool = remaining_df[
                ~remaining_df["filename"].isin(already_new)
            ].copy()

            filler_n = min(remaining_needed, len(filler_pool))

            if filler_n > 0:
                filler = filler_pool.sample(n=filler_n, random_state=RANDOM_STATE)
                new_selected_df = pd.concat([new_selected_df, filler], ignore_index=True)

        # Αν λόγω rounding ξεπεράσαμε, κόβουμε
        if len(new_selected_df) > need_n:
            new_selected_df = new_selected_df.sample(n=need_n, random_state=RANDOM_STATE)

        for _, row in new_selected_df.iterrows():
            selected_rows.append({
                "filename": row["filename"],
                "journal": row["journal"],
                "journal_short": row["journal_short"],
                "year": row["year"],
                "citation_bin": row["citation_bin"],
                "add_in_milestone": milestone_name,
                "milestone_pct": pct
            })

        previous_selected.update(new_selected_df["filename"].tolist())

sampling_plan = pd.DataFrame(selected_rows)

# milestone order
milestone_order = {name: i for i, name in enumerate(MILESTONES.keys(), start=1)}
sampling_plan["milestone_order"] = sampling_plan["add_in_milestone"].map(milestone_order)

sampling_plan = sampling_plan.sort_values(
    ["milestone_order", "journal", "year", "citation_bin", "filename"]
).reset_index(drop=True)

# =========================
# SAVE MAIN SAMPLING PLAN
# =========================

plan_path = os.path.join(OUT_DIR, "milestone_sampling_plan.csv")
sampling_plan.to_csv(plan_path, index=False)

print("Saved sampling plan:")
print(plan_path)

# =========================
# CREATE INCREMENTAL + CUMULATIVE FILELISTS
# =========================

for milestone_name in MILESTONES.keys():

    current_order = milestone_order[milestone_name]

    incremental = sampling_plan[
        sampling_plan["add_in_milestone"] == milestone_name
    ].copy()

    cumulative = sampling_plan[
        sampling_plan["milestone_order"] <= current_order
    ].copy()

    incremental_filelist_path = os.path.join(
        OUT_DIR,
        f"{milestone_name}_incremental_filelist.txt"
    )

    cumulative_filelist_path = os.path.join(
        OUT_DIR,
        f"{milestone_name}_cumulative_filelist.txt"
    )

    incremental["filename"].to_csv(
        incremental_filelist_path,
        index=False,
        header=False
    )

    cumulative["filename"].to_csv(
        cumulative_filelist_path,
        index=False,
        header=False
    )

# =========================
# SUMMARY TABLES
# =========================

summary_by_milestone_journal = (
    sampling_plan
    .groupby(["add_in_milestone", "journal", "journal_short"])
    .size()
    .reset_index(name="incremental_N")
)

summary_cumulative_rows = []

for milestone_name in MILESTONES.keys():
    current_order = milestone_order[milestone_name]

    cumulative = sampling_plan[
        sampling_plan["milestone_order"] <= current_order
    ]

    tmp = (
        cumulative
        .groupby(["journal", "journal_short"])
        .size()
        .reset_index(name="cumulative_N")
    )

    tmp["milestone"] = milestone_name
    summary_cumulative_rows.append(tmp)

summary_cumulative = pd.concat(summary_cumulative_rows, ignore_index=True)

journal_totals = (
    df.groupby(["journal", "journal_short"])
    .size()
    .reset_index(name="journal_total_low_high")
)

summary_cumulative = summary_cumulative.merge(
    journal_totals,
    on=["journal", "journal_short"],
    how="left"
)

summary_cumulative["actual_coverage"] = (
    summary_cumulative["cumulative_N"] /
    summary_cumulative["journal_total_low_high"]
)

summary_by_year_label = (
    sampling_plan
    .groupby(["add_in_milestone", "journal", "journal_short", "year", "citation_bin"])
    .size()
    .reset_index(name="incremental_N")
)

summary_by_milestone_journal.to_csv(
    os.path.join(OUT_DIR, "sampling_summary_incremental_by_journal.csv"),
    index=False
)

summary_cumulative.to_csv(
    os.path.join(OUT_DIR, "sampling_summary_cumulative_by_journal.csv"),
    index=False
)

summary_by_year_label.to_csv(
    os.path.join(OUT_DIR, "sampling_summary_by_journal_year_label.csv"),
    index=False
)

print("\nCumulative summary:")
display(summary_cumulative)

print("\nIncremental summary:")
display(summary_by_milestone_journal)

print("\nDone.")

In [ ]:
# =========================
# FINAL COVERAGE TABLE BY JOURNAL
# =========================

# Φτιάχνουμε τελικό πίνακα τύπου:
# Journal | Total low+high | v1 30% | v2 50% | v3 70% | v4 100%

coverage_table = summary_cumulative.pivot_table(
    index="journal",
    columns="milestone",
    values="cumulative_N",
    aggfunc="first"
).reset_index()

# Προσθέτουμε το συνολικό πλήθος low+high ανά journal
journal_totals_for_table = (
    summary_cumulative[["journal", "journal_total_low_high"]]
    .drop_duplicates()
)

coverage_table = coverage_table.merge(
    journal_totals_for_table,
    on="journal",
    how="left"
)

# Μετονομασία στηλών
coverage_table = coverage_table.rename(columns={
    "journal": "Journal",
    "journal_total_low_high": "Total low+high",
    "v1_coverage_30pct": "v1 30%",
    "v2_coverage_50pct": "v2 50%",
    "v3_coverage_70pct": "v3 70%",
    "v4_coverage_100pct": "v4 100%"
})

# Κρατάμε τη σωστή σειρά στηλών
coverage_table = coverage_table[
    [
        "Journal",
        "Total low+high",
        "v1 30%",
        "v2 50%",
        "v3 70%",
        "v4 100%"
    ]
]

# Ταξινόμηση αλφαβητικά ανά journal
coverage_table = coverage_table.sort_values("Journal").reset_index(drop=True)

# Προσθήκη γραμμής συνόλου
total_row = {
    "Journal": "Σύνολο",
    "Total low+high": coverage_table["Total low+high"].sum(),
    "v1 30%": coverage_table["v1 30%"].sum(),
    "v2 50%": coverage_table["v2 50%"].sum(),
    "v3 70%": coverage_table["v3 70%"].sum(),
    "v4 100%": coverage_table["v4 100%"].sum()
}

coverage_table = pd.concat(
    [coverage_table, pd.DataFrame([total_row])],
    ignore_index=True
)

# Εμφάνιση τελικού πίνακα
print("\nFinal coverage table by journal:")
display(coverage_table)

# Αποθήκευση
coverage_table_path = os.path.join(
    OUT_DIR,
    "final_coverage_table_by_journal.csv"
)

coverage_table.to_csv(coverage_table_path, index=False)

print("Saved final coverage table:")
print(coverage_table_path)